# Notebook 2 — Baseline Generation & Trace Logging
## Phi-4 (Azure) vs Llama 3.1 (Groq) — BIAT Financial Report Generation

**Purpose:** Generate model outputs for the golden evaluation case and log everything to MLflow.

### What this notebook does
1. Loads the registered system prompt from MLflow Prompt Registry (versioned)
2. Loads the golden test case from the MLflow Evaluation Dataset
3. Runs each model across 5 temperature settings (0.0, 0.1, 0.3, 0.5, 0.7)
4. Logs each generation as a traced MLflow child run
5. Saves all outputs to CSV and JSONL for Notebook 4

### Temperature variation rationale
Running at 5 temperatures instead of a single temperature=0.0 allows Notebook 4 to measure
**output stability**: a model with low score variance across temperatures is more reliable
in production than one whose quality fluctuates with sampling randomness.

### Output files
| File | Contents |
|---|---|
| `baseline_outputs_{CASE_ID}.csv` | 10 rows — one per model per temperature run |
| `baseline_outputs_{CASE_ID}.jsonl` | Same data in JSONL format |
| `baseline_outputs_{CASE_ID}_summary.csv` | 2 rows — per-model latency summary |

### This notebook does NOT
- Score outputs (that is Notebook 4's job)
- Compute similarity metrics
- Create charts

---
### Table of contents
1. Imports
2. Environment and path setup
3. MLflow experiment setup
4. Handoff values from Notebook 1
5. Load prompt and dataset
6. Section parser and normalization helpers
7. Model client setup
8. Generation function (traced)
9. Multi-temperature generation loop
10. Output preview
11. Notebook handoff summary

In [1]:
from __future__ import annotations

import os
import sys
import json
import time
from pathlib import Path
from typing import Any

import mlflow
import openai
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

from mlflow.entities import SpanType
from mlflow.genai.datasets import get_dataset

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ENV_PATH = PROJECT_ROOT / ".env"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(ENV_PATH)

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PHI4_API_KEY = os.getenv("PHI4_API_KEY")
PHI4_ENDPOINT = os.getenv("PHI4_ENDPOINT")
PHI4_DEPLOYMENT = os.getenv("PHI4_DEPLOYMENT")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_BASE_URL = os.getenv("BASE_URL")
GROQ_LLAMA_MODEL = os.getenv("GROQ_LLAMA_MODEL")

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME")

required_env = {
   # "OPENAI_API_KEY": OPENAI_API_KEY,
    "PHI4_API_KEY": PHI4_API_KEY,
    "PHI4_ENDPOINT": PHI4_ENDPOINT,
    "PHI4_DEPLOYMENT": PHI4_DEPLOYMENT,

    "GROQ_API_KEY": GROQ_API_KEY,
    "GROQ_BASE_URL": GROQ_BASE_URL,
    "GROQ_LLAMA_MODEL": GROQ_LLAMA_MODEL,

    "MLFLOW_TRACKING_URI": MLFLOW_TRACKING_URI,
    "MLFLOW_EXPERIMENT_NAME": MLFLOW_EXPERIMENT_NAME,
}

missing = [k for k, v in required_env.items() if not v]
if missing:
    raise EnvironmentError(f"Missing required environment variables: {missing}")

print("Environment validation passed.")
print("MLflow version:", mlflow.__version__)
print("OpenAI version:", openai.__version__)

Environment validation passed.
MLflow version: 3.10.1
OpenAI version: 2.29.0


In [3]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if experiment is None:
    raise RuntimeError("MLflow experiment could not be resolved.")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name)
print("Experiment ID:", experiment.experiment_id)

Tracking URI: http://127.0.0.1:5000
Experiment: phi4_vs_Llama3-1_baseline
Experiment ID: 4


## Enable MLflow autologging for OpenAI-compatible clients

In [4]:
mlflow.openai.autolog()

## Notebook 1 handoff values

In [5]:
PROMPT_URI_VERSIONED = "prompts:/financial_analysis_baseline/11"
DATASET_ID = "d-92231453cc9f4fbd83fa1978c64293bc"
CASE_ID = "test_case_004"
TEMPERATURES = [float(t) for t in os.getenv("RUN_TEMPERATURES", "0.0,0.1,0.3,0.5,0.7").split(",")]
N_RUNS = len(TEMPERATURES)


## Load prompt and dataset

In [6]:
prompt = mlflow.genai.load_prompt(PROMPT_URI_VERSIONED)
dataset = get_dataset(dataset_id=DATASET_ID)
dataset_df = dataset.to_df()

if dataset_df.empty:
    raise ValueError("The evaluation dataset is empty.")

print("Prompt name:", prompt.name)
print("Prompt version:", prompt.version)
print("Dataset ID:", dataset.dataset_id)
print("Dataset rows:", len(dataset_df))

Prompt name: financial_analysis_baseline
Prompt version: 11
Dataset ID: d-92231453cc9f4fbd83fa1978c64293bc
Dataset rows: 1


## parser

In [7]:
import re
import json

EXPECTED_SECTIONS = [
    "Executive Summary",
    "Profitability and Operational Efficiency",
    "Revenue Dynamics",
    "Asset Quality and Risk Profile",
    "Balance Sheet Structure and Liquidity",
    "Capital Adequacy",
    "Key Risks and Watch Points",
    "Conclusion",
]

SECTION_ALIASES = {
    "Executive Summary": ["Executive Summary"],
    "Profitability and Operational Efficiency": [
        "Profitability and Operational Efficiency",
        "Profitability and Efficiency",
        "Profitability & Efficiency",
    ],
    "Revenue Dynamics": ["Revenue Dynamics"],
    "Asset Quality and Risk Profile": [
        "Asset Quality and Risk Profile",
        "Risk Profile",
    ],
    "Balance Sheet Structure and Liquidity": [
        "Balance Sheet Structure and Liquidity",
        "Liquidity & Balance Sheet",
        "Liquidity and Balance Sheet",
    ],
    "Capital Adequacy": ["Capital Adequacy"],
    "Key Risks and Watch Points": [
        "Key Risks and Watch Points",
        "Risks & Watch Points",
        "Risks and Watch Points",
    ],
    "Conclusion": ["Conclusion"],
}

def normalize_report_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[–—−]", "-", text)
    # Remove bold markdown formatting (**text** -> text)
    text = re.sub(r"\*\*(.+?)\*\*", r"\1", text)
    return text

def extract_sections_rule_based(text: str) -> dict[str, str]:
    text = normalize_report_text(text)
    results = {name: "" for name in EXPECTED_SECTIONS}
    matches = []

    for canonical, aliases in SECTION_ALIASES.items():
        for alias in aliases:
            pattern = rf"(?im)^[ \t]*(?:#+[ \t]*)?{re.escape(alias)}[ \t]*:?[ \t]*$"
            for m in re.finditer(pattern, text):
                matches.append((m.start(), m.end(), canonical))

    matches.sort(key=lambda x: x[0])

    seen = set()
    ordered = []
    for item in matches:
        key = (item[0], item[2])
        if key not in seen:
            ordered.append(item)
            seen.add(key)

    for i, (_, end_pos, canonical) in enumerate(ordered):
        next_start = ordered[i + 1][0] if i + 1 < len(ordered) else len(text)
        results[canonical] = text[end_pos:next_start].strip()

    return results

def structure_components(text: str) -> dict:
    sections = extract_sections_rule_based(text)
    present = {k: int(bool(v.strip())) for k, v in sections.items()}
    missing_sections = [k for k, v in sections.items() if not v.strip()]
    structure_score = sum(present.values()) / len(EXPECTED_SECTIONS)

    return {
        **{f"has_{k.lower().replace(' ', '_').replace('&', 'and').replace('-', '_')}": v for k, v in present.items()},
        "structure_score": structure_score,
        "missing_sections_count": len(missing_sections),
        "missing_sections": json.dumps(missing_sections, ensure_ascii=False),
        "parsed_sections_json": json.dumps(sections, ensure_ascii=False),
    }

## Helper to normalize dataset cells

Guarantees:
- always returns a dict
- no matter what MLflow gave you

> we normalize dataset cells because MLflow does not guarantee that inputs and expectations will always be Python dictionaries after to_df().

In [8]:
def ensure_dict(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError as e:
            raise ValueError(f"Invalid JSON string: {value}") from e
    raise TypeError(f"Expected dict or JSON string, got {type(value)}")

In [9]:
dataset_df["inputs_parsed"] = dataset_df["inputs"].apply(ensure_dict)
dataset_df["expectations_parsed"] = dataset_df["expectations"].apply(ensure_dict)

In [10]:
matches = dataset_df[
    dataset_df["inputs_parsed"].apply(lambda x: x.get("case_id") == CASE_ID)
]

if matches.empty:
    raise ValueError(f"CASE_ID '{CASE_ID}' not found in dataset.")

record = matches.iloc[0]

inputs = record["inputs_parsed"]
expectations = record["expectations_parsed"]

financial_data = inputs["financial_data"]
reference_output = expectations["expected_response"]

print("Loaded case:", inputs["case_id"])
print("Financial data length:", len(financial_data))
print("Reference output length:", len(reference_output))

Loaded case: test_case_004
Financial data length: 1405
Reference output length: 4706


##  Build formatted messages from the registered prompt

In [11]:
messages = prompt.format(financial_data=financial_data)

if not isinstance(messages, list):
    raise TypeError("Expected a chat prompt that formats into a list of messages.")

print("Formatted message count:", len(messages))
print(messages[0])

Formatted message count: 2
{'role': 'system', 'content': 'You are a financial analyst specialized in bank performance analysis.\n\nYour task is to analyze only the financial data provided by the user and write a professional, grounded, and well-structured financial analysis report.\n\nStrict rules:\n1. Base your answer only on the provided data.\n2. Do not invent numbers, ratios, trends, explanations, or unsupported claims.\n3. Do not rename, merge, omit, or reorder section titles.\n4. Use exactly the following section titles, in exactly this order:\n   Executive Summary\n   Profitability and Operational Efficiency\n   Revenue Dynamics\n   Asset Quality and Risk Profile\n   Balance Sheet Structure and Liquidity\n   Capital Adequacy\n   Key Risks and Watch Points\n   Conclusion\n5. Every section must appear in the final answer, even if brief.\n6. If the provided data is insufficient for a section, write exactly: Not enough information in the provided data.\n7. Use concise analytical par

## Create provider clients

In [12]:
# gpt35_client = OpenAI(api_key=OPENAI_API_KEY)

# Azure/OpenAI-compatible endpoint for Phi-4
phi4_client = OpenAI(
    api_key=PHI4_API_KEY,
    base_url=PHI4_ENDPOINT,
)

# Groq API client for LLaMA 3
groq_client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url=GROQ_BASE_URL,
)

MODEL_CONFIGS = [
    {
        "client": phi4_client,
        "provider": "azure_openai_compatible",
        "model_label": "Phi-4",
        "model_name": PHI4_DEPLOYMENT,
    },
    {
        "client": groq_client,
        "provider": "groq",
        "model_label": "Llama 3.1",
        "model_name": GROQ_LLAMA_MODEL,
    },
]


## Response helpers

In [13]:
def extract_text(response: Any) -> str:
    try:
        return response.choices[0].message.content.strip()
    except Exception as e:
        raise ValueError(f"Could not extract response text: {e}") from e


def extract_usage_dict(response: Any) -> dict[str, Any]:
    usage = getattr(response, "usage", None)
    if usage is None:
        return {
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": None,
        }

    if hasattr(usage, "model_dump"):
        raw = usage.model_dump()
    elif isinstance(usage, dict):
        raw = usage
    else:
        raw = {
            "prompt_tokens": getattr(usage, "prompt_tokens", None),
            "completion_tokens": getattr(usage, "completion_tokens", None),
            "total_tokens": getattr(usage, "total_tokens", None),
        }

    return {
        "prompt_tokens": raw.get("prompt_tokens"),
        "completion_tokens": raw.get("completion_tokens"),
        "total_tokens": raw.get("total_tokens"),
    }

## Traced generation function

In [14]:
@mlflow.trace(name="generate_with_model", span_type=SpanType.CHAIN)
def generate_with_model(
    *,
    client: OpenAI,
    provider: str,
    model_label: str,
    model_name: str,
    prompt_uri: str,
    financial_data: str,
    case_id: str,
    temperature: float = 0.0,
    max_tokens: int = 1200,
) -> dict[str, Any]:
    # Load inside the traced function so the prompt is linked to the trace
    traced_prompt = mlflow.genai.load_prompt(prompt_uri)
    traced_messages = traced_prompt.format(financial_data=financial_data)

    start = time.perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=traced_messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    latency_sec = time.perf_counter() - start

    output_text = extract_text(response)
    usage_dict = extract_usage_dict(response)

    return {
        "case_id": case_id,
        "provider": provider,
        "model_label": model_label,
        "model_name": model_name,
        "prompt_uri": prompt_uri,
        "prompt_name": traced_prompt.name,
        "prompt_version": traced_prompt.version,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "latency_sec": round(latency_sec, 4),
        "output_text": output_text,
        "prompt_tokens": usage_dict["prompt_tokens"],
        "completion_tokens": usage_dict["completion_tokens"],
        "total_tokens": usage_dict["total_tokens"],
    }

In [15]:
with mlflow.start_run(run_name="nb2_baseline_generation_biat") as run:
    run_id = run.info.run_id

    mlflow.log_params({
        "phase": "generation",
        "case_id": CASE_ID,
        "prompt_uri": PROMPT_URI_VERSIONED,
        "dataset_id": DATASET_ID,
        "dataset_rows": len(dataset_df),
        "phi4_model_name": PHI4_DEPLOYMENT,
        #"gpt_model_name": "gpt-3.5-turbo",
        "llama_model_name": GROQ_LLAMA_MODEL,
        "n_runs": N_RUNS,
        "temperatures": str(TEMPERATURES),
    })

    mlflow.set_tags({
        "notebook": "nb2",
        "stage": "generation",
        "comparison": "phi4_vs_llama31",
    })

    all_run_rows = []

    for run_idx, temp in enumerate(TEMPERATURES):
        for cfg in MODEL_CONFIGS:
            with mlflow.start_run(
                run_name=f"{cfg['model_label']}_temp_{temp}",
                nested=True,
                tags={
                    "run_temperature": str(temp),
                    "run_idx": str(run_idx),
                    "model_label": cfg["model_label"],
                },
            ):
                result = generate_with_model(
                    client=cfg["client"],
                    provider=cfg["provider"],
                    model_label=cfg["model_label"],
                    model_name=cfg["model_name"],
                    prompt_uri=PROMPT_URI_VERSIONED,
                    financial_data=financial_data,
                    case_id=CASE_ID,
                    temperature=temp,
                )
                result["run_temperature"] = temp
                result["run_seed"] = run_idx
                all_run_rows.append(result)
                mlflow.log_metric("run_temperature", temp)

    outputs_df = pd.DataFrame(all_run_rows)
    outputs_df["reference_output"] = reference_output
    outputs_df["financial_data"] = financial_data

    outputs_df["output_text_normalized"] = outputs_df["output_text"].apply(normalize_report_text)
    outputs_df["reference_output_normalized"] = reference_output

    output_structure_df = outputs_df["output_text"].apply(structure_components).apply(pd.Series)
    reference_structure = structure_components(reference_output)

    outputs_df = pd.concat([outputs_df, output_structure_df], axis=1)
    outputs_df["reference_sections_json"] = reference_structure["parsed_sections_json"]

    outputs_summary_df = (
        outputs_df.groupby("model_label", as_index=False)["latency_sec"]
        .agg(mean_latency_sec="mean", std_latency_sec="std")
    )

    csv_path = OUTPUTS_DIR / f"baseline_outputs_{CASE_ID}.csv"
    summary_csv_path = OUTPUTS_DIR / f"baseline_outputs_{CASE_ID}_summary.csv"
    jsonl_path = OUTPUTS_DIR / f"baseline_outputs_{CASE_ID}.jsonl"

    outputs_df.to_csv(csv_path, index=False, encoding="utf-8")
    outputs_summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8")
    outputs_df.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)

    mlflow.log_metric("n_runs_per_model", N_RUNS)
    mlflow.log_metric("total_outputs", len(outputs_df))
    mlflow.log_artifact(str(csv_path), artifact_path="generated_outputs")
    mlflow.log_artifact(str(summary_csv_path), artifact_path="generated_outputs")
    mlflow.log_artifact(str(jsonl_path), artifact_path="generated_outputs")
    mlflow.log_text(reference_output, "generated_outputs/reference_output.txt")
    mlflow.log_text(financial_data, "generated_outputs/financial_input.txt")

print("Run ID:", run_id)


🏃 View run Phi-4_temp_0.0 at: http://127.0.0.1:5000/#/experiments/4/runs/cff420f5a7fc4fc59c1ed28401ac0d94
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
🏃 View run Llama 3.1_temp_0.0 at: http://127.0.0.1:5000/#/experiments/4/runs/5c3a205e41ad40fc80ac114fdc33aedf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
🏃 View run Phi-4_temp_0.1 at: http://127.0.0.1:5000/#/experiments/4/runs/6b32e70b57b84eb285ae9d23ab32d6c2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
🏃 View run Llama 3.1_temp_0.1 at: http://127.0.0.1:5000/#/experiments/4/runs/b0d1846342b143b28046ba9be131ac3f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
🏃 View run Phi-4_temp_0.3 at: http://127.0.0.1:5000/#/experiments/4/runs/39fa9e408ccb4def8c1b90d24b667b07
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
🏃 View run Llama 3.1_temp_0.3 at: http://127.0.0.1:5000/#/experiments/4/runs/e1c45af908424ceaa03c4aca0baf3d7e
🧪 View experiment at: http://127.0.0.1:5000/#/experi

[Trace(trace_id=tr-bacf158a0f69306abb9edf8b0639d4f0), Trace(trace_id=tr-91e1b2329a31465f523d33e7a379d68b), Trace(trace_id=tr-88f4b556a46136d0e172850b2bfd0b78), Trace(trace_id=tr-8140b9798ef24e23ee338be891d31947), Trace(trace_id=tr-9c0b744b32a2d1eb25bd428ec6f88c87), Trace(trace_id=tr-7705f11be16ffcf041a09ed6ec419c23), Trace(trace_id=tr-3464c7a7a7a095a07fade7fed59dbfd4), Trace(trace_id=tr-804e750bf00538691e5d73acd57e9ed8), Trace(trace_id=tr-0d4917af88f816b387f3e0ee022bdb3b), Trace(trace_id=tr-0c57ee957f376f32213006dda3e6e94a)]

In [16]:
display(
    outputs_df[
        [
            "case_id",
            "model_label",
            "model_name",
            "temperature",
            "latency_sec",
            "prompt_tokens",
            "completion_tokens",
            "total_tokens",
            "output_text",
        ]
    ]
)

,case_id,model_label,model_name,temperature,latency_sec,prompt_tokens,completion_tokens,total_tokens,output_text
0,test_case_004,Phi-4,Phi-4-experimentation-investment-nextgen,0.0,22.6712,956,736,1692,**Executive Summary**\n\nBanque Internationale...
1,test_case_004,Llama 3.1,llama-3.1-8b-instant,0.0,1.3880,981,742,1723,Executive Summary\n\nBanque Internationale Ara...
2,test_case_004,Phi-4,Phi-4-experimentation-investment-nextgen,0.1,27.4613,956,829,1785,**Executive Summary**\n\nBanque Internationale...
3,test_case_004,Llama 3.1,llama-3.1-8b-instant,0.1,5.0886,981,857,1838,Executive Summary\n\nBanque Internationale Ara...
4,test_case_004,Phi-4,Phi-4-experimentation-investment-nextgen,0.3,23.8216,956,930,1886,**Executive Summary**\n\nBanque Internationale...
5,test_case_004,Llama 3.1,llama-3.1-8b-instant,0.3,6.7124,981,691,1672,Executive Summary\n----------------\n\nBanque ...
6,test_case_004,Phi-4,Phi-4-experimentation-investment-nextgen,0.5,23.8352,956,916,1872,**Executive Summary**\n\nBanque Internationale...
7,test_case_004,Llama 3.1,llama-3.1-8b-instant,0.5,1.2795,981,822,1803,Executive Summary\n\nBanque Internationale Ara...
8,test_case_004,Phi-4,Phi-4-experimentation-investment-nextgen,0.7,21.1765,956,888,1844,**Executive Summary**\n\nBanque Internationale...
9,test_case_004,Llama 3.1,llama-3.1-8b-instant,0.7,1.2909,981,657,1638,Executive Summary\n----------------\n\nThe Ban...


In [ ]:
# Preview outputs — shows first 400 chars per run to avoid flooding the notebook
print(f"{'─'*80}")
print(f"OUTPUT PREVIEW ({len(outputs_df)} rows — {N_RUNS} runs × {len(MODEL_CONFIGS)} models)")
print(f"{'─'*80}")

for _, row in outputs_df.iterrows():
    preview = row["output_text"][:400].replace("\n", " ")
    print(f"\n[{row['model_label']}] temp={row['temperature']} | "
          f"{row['completion_tokens']} tokens | {row['latency_sec']}s")
    print(f"  {preview}{'...' if len(row['output_text']) > 400 else ''}")

In [18]:
nb2_summary = {
    "run_id": run_id,
    "prompt_uri": PROMPT_URI_VERSIONED,
    "dataset_id": DATASET_ID,
    "case_id": CASE_ID,
    "output_csv": str(csv_path),
    "output_jsonl": str(jsonl_path),
    "output_summary_csv": str(summary_csv_path),
    "rows_generated": len(outputs_df),
    "temperatures": TEMPERATURES,
    "n_runs_per_model": N_RUNS,
    
}

print(json.dumps(nb2_summary, indent=2))

{
  "run_id": "aa8afc17af474639a44f02100e7a366f",
  "prompt_uri": "prompts:/financial_analysis_baseline/11",
  "dataset_id": "d-92231453cc9f4fbd83fa1978c64293bc",
  "case_id": "test_case_004",
  "output_csv": "c:\\Users\\BRHN\\Desktop\\SLM-evals\\outputs\\baseline_outputs_test_case_004.csv",
  "output_jsonl": "c:\\Users\\BRHN\\Desktop\\SLM-evals\\outputs\\baseline_outputs_test_case_004.jsonl",
  "output_summary_csv": "c:\\Users\\BRHN\\Desktop\\SLM-evals\\outputs\\baseline_outputs_test_case_004_summary.csv",
  "rows_generated": 10,
  "temperatures": [
    0.0,
    0.1,
    0.3,
    0.5,
    0.7
  ],
  "n_runs_per_model": 5
}


In [19]:
mlflow.end_run()